# Домашнее задание: RAG

В рамках домашнего задания вам предстоит построить RAG-систему из различных компонент для задачи ускорения поиска информации пользователем в разделе helpme сервиса.

Сегодня бизнес пришел к вам со следующей задачей: пользователи различных сервисов mail.ru ежедневно задают тысячи вопросов, многие из которых имеют стандартные решения. Но на странице helpme ответы на такие вопросы приходится долго и сложно искать, поэтому пользователи решают задать вопрос в службу поддержки, что отнимает кучу денег у бизнеса. Для решения этой задачи было решено построить RAG-систему, которая будет по вопросам пользователей искать подходящий документ среди страниц help me, и генерировать ответ на основе этого документа, тем самым экономя время пользователей и ресурсы поддержки.

Для данного кейса мы стандартными скраперами обкачали страницы раздела helpme сервисов mail.ru: https://help.mail.ru/. Пользовательские запросы и разметки получены с помощью LLM.

In [1]:
# отдельные версии библиотек, с которыми осуществлялся запуск, можно подсмотреть в requirements.txt
# но в колабе надежнее запустить просто команды ниже

!pip install langchain==1.2.4 langchain-classic==1.0.1 langchain-community==0.4.1 langchain-core==1.2.7 langchain-openai==1.1.7 langchain-text-splitters==1.1.0 together==1.5.35
!pip install faiss-cpu sentence-transformers requests transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.9/268.9 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully unin

# Читаем данные

In [4]:
# https://help.mail.ru/

import json
docs_path = "/content/drive/MyDrive/llm_course/helpme_data_markdawn.json"
queries_path = "/content/drive/MyDrive/llm_course/test_questions_100.txt"
with open(docs_path, "r") as fin:
    pages = json.load(fin)

with open(queries_path, "r") as fin:
    queries = fin.read().strip().split('\n\n')
    queries = list(map(lambda w: w.strip(), queries))

print(len(pages), len(queries))
print(pages[20][:100], '...\n\n', '-' * 30, "\n\n", "\n".join(queries[:3]))

2119 101
# Проблема с Облаком

Если выне можете авторизоваться, воспользуйтесь рекомендациями из статьи «Не м ...

 ------------------------------ 

 Я забыл и пароль, и ответ на вопрос — как теперь войти?
Почему письма от нужного адреса постоянно попадают в спам?
Не приходят уведомления на телефон, что делать?


In [5]:
pages[21]

'# Что такое Облако Mail и как им пользоваться?\n\n## Подписка Mail Space — попробуйте, с ней возможностей больше\n\n## Подписка Mail Space — попробуйте\n\nОблако Mail — это ваше персональное надежное хранилище в интернете. Все нужные файлы всегда под рукой, доступны вам в любой точке мира с любого устройства. С помощью Облака вы экономите место на жестком диске компьютера или в памяти смартфона. В Облаке 8 ГБ предоставляется бесплатно, но вы можете купить дополнительную память.\n\n## Нужные файлы всегда под рукой\n\nОблако Mail работает на всех самых популярных десктопных и мобильных платформах: Windows, MacOS, Android, iOS. Вы сможете пользоваться преимуществами Облака вне зависимости от того, какое у вас устройство.\n\n## Высокая скорость загрузки и скачивания файлов\n\nФайлы в Облако заливаются за считанные секунды. Мы не ограничиваем скоростьзагрузкиискачивания, она зависит только от способа вашего интернет-подключения.\n\n## Надежное хранилище для всех ваших файлов. Резервные коп

# MVP - 10 баллов

Соберите MVP-решение с использованием langchain и векторным поиском на основе FAISS. В качестве векторного поиска можете воспользоваться FAISS sentence-transformers/all-MiniLM-L6-v2, в качестве ллм -- api [together_ai](https://www.together.ai/) и моделью Qwen/Qwen2.5-7B-Instruct-Turbo

Для решения вам понадобится написать короткий код, правильно использовав из фреймворка langchain компоненты HuggingFaceEmbeddings, FAISS, Together, Document, изучив официальную документацию.

Какие-то решения можете подсмотреть по следующей статье:
https://python.langchain.com/docs/tutorials/rag/

In [6]:
# для тестирования возьмем пару документов, придумаем вопросы по ним и посмотрим, справляется ли LLM
pages_sample = pages[20:22]
queries_sample = [
    "Списываются деньги за тариф, которым я не пользуюсь — как это отключить?", # по 20му доку
    "На телефоне с Android не вижу свои файлы в Облаке — почему так?" # по 21му доку
]

In [6]:
# пример вызова together ai
from together import Together

client = Together(
    api_key=api_key,
    base_url="https://litellm.deepschool.ru",
)
result = client.chat.completions.create(
  model="Qwen/Qwen2.5-7B-Instruct-Turbo",
  messages=[{"role": "user", "content": "What are the top 3 things to do in New York?"}],
  max_tokens=10
)
print(result.choices[0].message.content)

In [7]:
from together import Together
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# ---- Ваш код здесь ----
print("ваш код здесь")
# ---- Конец кода ----



╭───────────────────────────────────────────── 🚀 New SDK Available ──────────────────────────────────────────────╮
│ Together Python SDK 2.0 is now available!                                                                       │
│                                                                                                                 │
│ Install the beta:                                                                                               │
│ pip install --pre together  or  uv add together --prerelease allow                                              │
│                                                                                                                 │
│ New SDK: ]8;id=596368;https://github.com/togethercomputer/together-py\https://github.com/togethercomputer/together-py]8;;\                                                        │
│ Migration guide: ]8;id=178241;https://docs.together.ai/docs/pythonv2-migration-guide\https://docs.together.ai/docs/pythonv2-migration-guide]8;;\                                         │
│                                                                                                                 │
│ This package will be maintained until January 2026.                                                             │
│ Set TOGETHER_NO_BANNER=1 to hide this message.                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/tmp/ipython-input-1800723147.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnin

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Вопрос: Списываются деньги за тариф, которым я не пользуюсь — как это отключить?
Ответ: Если у вас списываются деньги за тариф, которым вы не пользуетесь, вы можете воспользоваться рекомендациями из статьи «У меня списываются деньги за тариф, который мне не нужен». Однако, основываясь на предоставленном контексте, конкретные шаги для отключения тарифа не указаны. Для получения точных инструкций вам следует перейти в соответствующую статью на официальном сайте или в приложении.

Вопрос: На телефоне с Android не вижу свои файлы в Облаке — почему так?
Ответ: Не знаю по предоставленному контекstu. Контекст не содержит информации о том, что может быть неправильно отображено на телефоне с Android в Облаке Mail.



Данное решение можно настроить очень быстро и оно уже может показывать приемлемое качество. Более того, для ряда задач такое решение будет достаточным, но для большинства задач мы все же увидим тут проблемы с качеством поиска, генераций и масштабируемостью такого решения.

# Разложим по компонентам - 15 баллов

API это хорошо, но для большей гибкости и масштабируемости решения нам рано или поздно может понадобиться перейти на локальный инференс.

Напишем родительский класс BaseGenerator, который будет иметь метод ```generate_text(self, prompt: str, enable_thinking: bool = False)```, принимать на вход произвольный prompt и параметр, отвечающий за необходимость ризонинга в LLM (параметр инференса Qwen3).

От этого класса будем наследовать все генераторы и реализовывать в них методы ```get_prompt```, который формирует по необходимому входу строку промпта, включающего входные данные, и ```generate```, который будет получать необходимый промпт и вызывать метод ```generate_text```

Подсмотреть инференс квена можно [здесь](https://huggingface.co/Qwen/Qwen3-1.7B)

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

class BaseGenerator:
    def __init__(self, model_name: str = "Qwen/Qwen3-1.7B", max_new_tokens: int = 3000):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
        self.max_new_tokens = max_new_tokens

    @torch.no_grad()
    def generate_text(self, prompt: str, enable_thinking: bool = False) -> str:
        messages = [{"role": "user", "content": prompt}]

        # ---- Ваш код здесь ----
        print("ваш код здесь")
        # ---- Конец кода ----


        return {"thinking_content": thinking_content, "content": content}


In [9]:
class Generator(BaseGenerator):
    def get_prompt(self, query: str, docs: list[str]) -> str:

        # ---- Ваш код здесь ----
        print("ваш код здесь")
        # ---- Конец кода ----


    def generate(self, query: str, docs: list[str], enable_thinking: bool = False) -> str:
        prompt = self.get_prompt(query, docs)
        return self.generate_text(prompt, enable_thinking)


In [10]:
model_name = "Qwen/Qwen3-1.7B"
generator = Generator(model_name)
generator.generate("что за порода лабрабубель", ["многие не знают, что порода лабрабубель получилась в результате смесь лабрадора и бубеля"])

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/622M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

{'thinking_content': '',
 'content': 'Порода лабрабубель — это результат смешения лабрадора и бубеля. Она получила название из-за характерного внешнего вида собак, которые часто имеют тонкую, гладкую шерсть и небольшую голову. Эта порода не является официальной, но популярна среди собакодоров из-за своих уникальных черт.'}

Напишем чуть более гибкий и немного улучшенный пайплайн, в котором мы разбиваем документы на чанки с помощью RecursiveCharacterTextSplitter (размер чанка 500, перекрытие 100. В данной задаче также логично делить на чанки по структуре -- можно сделать по желанию); составлять индекс из чанков с помощью faiss.IndexFlatL2; а генерации получаем с помощью generator.generate(query, top_contexts), где top_contexts это топ-N (N=20) самых релевантных документов в терминах нашего векторного поиска.

Протестируем работу на нашем небольшом семпле.

In [12]:
import faiss
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import Callable


# ---- Ваш код здесь ----
print("ваш код здесь")

class SearchIndex:
    def __init__(self, tokenizer: Callable, chunk_size: int = 500, chunk_overlap: int = 100,
                 embed_model:str = "sentence-transformers/all-MiniLM-L6-v2"):
        pass

    def build_index(self, pages: list[str]) -> None:
        pass

    def search(self, query: str) -> list[str]:
        pass

# ---- Конец кода ----



In [13]:
search_index = SearchIndex(generator.tokenizer)
search_index.build_index(pages_sample)

# тестируем
for query in queries_sample:
    top_contexts = search_index.search(query)
    output = generator.generate(query, top_contexts)["content"].strip()
    print(f"Вопрос: {query}\nОтвет: {output}\n")

Вопрос: Списываются деньги за тариф, которым я не пользуюсь — как это отключить?
Ответ: Как отключить письма с вложениями, которые вы не используете, вы можете:

1. **Удалить вложения из писем**:  
   - Откройте письмо, в котором содержатся вложения.  
   - Нажмите на кнопку «Прикрепить файл» или «Вложения».  
   - Выберите файл, который хотите удалить, и нажмите «Удалить».  
   - Повторите процесс для всех писем с вложениями.

2. **Удалить письма из Облака**:  
   - Перейдите в раздел «Облако» в почту.  
   - Найдите письма, которые вы хотите удалить.  
   - Нажмите на «Удалить» или «Удалить все».  
   - Подтвердите удаление.

3. **Удалить письма из почты**:  
   - Откройте письма, которые вы хотите удалить.  
   - Нажмите на «Удалить» или «Удалить все».  
   - Подтвердите удаление.

4. **Увеличить место в Облаке**:  
   - Перейдите в раздел «Проблема с подпиской» и выберите тариф Mail Space.  
   - Увеличьте объем хранилища, чтобы не удалять письма.

5. **Отключить подписку**:  
   -

# Рефраз - 5 баллов

Запрос пользователя часто бывает непонятным: нечеткие формулировки, обрывистые фразы, двусмысленность. Для повышения качества имеет смысл перефразировать запрос.

На практике это обычно делается в два этапа:
1) берется какая-то подходящая по ресурсам LLM, к ней пишется промпт -- для старта работает ок
2) для улучшения действуют по понятной классической схеме -- берут большую умную модель (gpt-4o, deepseek и тп), пишут таргеты для запросов из целевого распределения, делают sft (при желании можно еще небольшой алайнмент) модели из п1 на эти таргеты. После этого качество становится получше.

Предлагается реализовать первый этап -- завести перефразирование на промпте. Как правило, для рефразера хватает LLM с меньшим числом параметром, нежели в основном генераторе.

In [14]:
class QueryRephraser(BaseGenerator):
    def get_prompt(self, query: str) -> str:
        # ---- Ваш код здесь ----
        print("ваш код здесь")
        # return ""
        # ---- Конец кода ----



    def rephrase(self, query: str, enable_thinking: bool = False) -> str:
        prompt = self.get_prompt(query)
        result = self.generate_text(prompt, enable_thinking)
        return result["content"]


In [15]:
# Рефразинг

# ---- Ваш код здесь ----
model_name = "" # выберите подходящего размера, например из семейства Qwen3
# ---- Конец кода ----

rephraser = QueryRephraser(model_name)
ref = rephraser.rephrase("погода москва завтра")
print("Рефразированный запрос:", ref)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Рефразированный запрос: Погода в Москве завтра.


# Реранк - 10 баллов

Документы, наиболее близкие в терминах векторного поиска, необязательно в том же порядке являются самыми полезными для генеративного ответа. Поэтому классическая схема -- взять топ-K документов, и переранжировать их в соответствии со своими нуждами. Как правило, такие ранкеры обучаются на спецсигналы, специфичные для продукта, либо напрямую через асессорскую/LLM-разметку.

Для первого приближения можно взять открытый мультиязычный ранкер Qwen3. Предлагается написать код, который позволит паре запрос+документ сопоставить скор, по которому можно эти документы ранжировать для запроса. Тогда последовательность будет выглядеть следующим образом: берем запрос, берем top_K релевантных ему документов, каждому документу ранкером сопоставляем скор, переранжируем документы, выбираем топ-1 документ.

Принцип вычисления скора: пишем промпт, в котором просим посмотреть на запрос, документ и ответить (в какой-то формулировке) на вопрос: правда ли, что этот документ релевантен запросу (стоит раскрыть, что такое релевантен), причем в качестве ответа можно генерировать только либо токен yes, либо токен no. Далее, LLM генерирует ответ, мы берем первый токен, смотрим только на логиты токенов yes и no, сводим эти предсказания к вероятности положительного ответа. Эта вероятность и будет нашим ранжирующим скором -- чем больше скор, тем более релевантен документ запросу.

Подсмотреть код для инференса можно [тут](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B)

In [16]:
# Requires transformers>=4.51.0
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM

class Reranker:
    def __init__(self, model_name: str = "Qwen/Qwen3-Reranker-0.6B"):
        # ---- Ваш код здесь ----
        print("ваш код здесь")
        # ---- Конец кода ----



    def get_scores(self, query: str, docs: list[str]):
        # ---- Ваш код здесь ----
        print("ваш код здесь")
        # ---- Конец кода ----

        return scores # для каждой пары (запрос, документ) для списка документов возвращаем скор


In [17]:
reranker = Reranker("Qwen/Qwen3-Reranker-0.6B")
reranker.get_scores(queries_sample[0], pages_sample)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2919: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


[0.9548653960227966, 0.00012889217759948224]

# Собираем всё вместе -- 10 баллов

Мы на финишной прямой -- осталось лишь собрать все написанные кусочки в единый пайплайн, разобравшись с порядком и вычислительными ресурсами. Возьмите набор запросов, постройте базу с чанками, запустите RAG-пайплайн для каждого запроса из списка.

In [18]:
# ---- Ваш код здесь ----
print("ваш код здесь")
# ---- Конец кода ----




Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Вопрос: Списываются деньги за тариф, которым я не пользуюсь — как это отключить?
Ответ: Списание денег за тариф, который не используется, не допускается. В соответствии с политикой оператора, только те тарифы, которые активно используются, подлежат оплате. Если вы не пользуетесь каким-либо тарифом, то не нужно платить за него. Для получения дополнительной информации рекомендуется обратиться в службу поддержки или проверить официальные документы оператора.


Вопрос: На телефоне с Android не вижу свои файлы в Облаке — почему так?
Ответ: Если уведомления не приходят в приложении Почты на Android, напишите нам на mailapps@corp.mail.ru.


